# Aims
- to implement the QJL part, i need to dequantise and unrotate the embeddings
- This is so that I can calculate the residuals and also the QJL signs

In [41]:
import numpy as np
import polars as pl
import pickle as pkl
import sys
sys.path.append("..")
from lib.turboquant.turboquant import dequantize_embeddings_polar, calculate_centroids_lookup_table

In [42]:
with open("../data/codebook/384d_codebook.pkl", "rb") as f:
    codebook = pkl.load(f)

In [43]:
e = pl.read_parquet("../data/processed/arxiv_quantized_embeddings_4bits_qjl.parquet")

In [44]:
e

id,packed_embedding,gamma,packed_qjl_bits,residuals,recovered_embedding,embedding
str,"array[u8, 256]",f32,"array[u8, 64]","array[f32, 384]","array[f32, 384]","array[f32, 384]"
"""0704.0001""","[31, 31, … 31]",2.813821,"[0, 0, … 0]","[-1.763473, 1.957815, … -0.049717]","[-1.922151, 1.947271, … 0.0]","[-0.158678, -0.010544, … 0.049717]"
"""0704.0002""","[31, 31, … 31]",2.897824,"[0, 0, … 0]","[-1.933493, 1.913186, … -0.021261]","[-1.922151, 1.947271, … 0.0]","[0.011342, 0.034085, … 0.021261]"
"""0704.0003""","[31, 31, … 31]",2.954786,"[0, 0, … 0]","[-1.95329, 1.979244, … -0.004312]","[-1.922151, 1.947271, … 0.0]","[0.031139, -0.031973, … 0.004312]"
"""0704.0004""","[31, 31, … 31]",2.879711,"[0, 0, … 0]","[-1.8533, 1.965473, … -0.000149]","[-1.922151, 1.947271, … 0.0]","[-0.068851, -0.018202, … 0.000149]"
"""0704.0005""","[31, 31, … 31]",2.934319,"[0, 0, … 0]","[-1.957704, 1.943937, … 0.026612]","[-1.922151, 1.947271, … 0.0]","[0.035553, 0.003334, … -0.026612]"
…,…,…,…,…,…,…
"""0802.3617""","[31, 31, … 31]",2.858917,"[0, 0, … 0]","[-1.909659, 1.879203, … -0.032157]","[-1.922151, 1.947271, … 0.0]","[-0.012492, 0.068068, … 0.032157]"
"""0802.3618""","[31, 31, … 31]",2.897864,"[0, 0, … 0]","[-1.889797, 1.956377, … -0.009674]","[-1.922151, 1.947271, … 0.0]","[-0.032354, -0.009106, … 0.009674]"
"""0802.3619""","[31, 31, … 31]",2.87836,"[0, 0, … 0]","[-1.814137, 2.002135, … 0.020686]","[-1.922151, 1.947271, … 0.0]","[-0.108015, -0.054864, … -0.020686]"


In [45]:
lut = calculate_centroids_lookup_table(codebook, 4)

In [46]:
lut

array([[-0.12091945, -0.12091945],
       [-0.12091945, -0.08460474],
       [-0.12091945, -0.06134046],
       [-0.12091945, -0.04448182],
       [-0.12091945, -0.03125859],
       [-0.12091945, -0.01997488],
       [-0.12091945, -0.00952856],
       [-0.12091945,  0.00078499],
       [-0.12091945,  0.0112464 ],
       [-0.12091945,  0.02168263],
       [-0.12091945,  0.0320071 ],
       [-0.12091945,  0.04226933],
       [-0.12091945,  0.05339511],
       [-0.12091945,  0.06735399],
       [-0.12091945,  0.08780586],
       [-0.12091945,  0.12175898],
       [-0.08460474, -0.12091945],
       [-0.08460474, -0.08460474],
       [-0.08460474, -0.06134046],
       [-0.08460474, -0.04448182],
       [-0.08460474, -0.03125859],
       [-0.08460474, -0.01997488],
       [-0.08460474, -0.00952856],
       [-0.08460474,  0.00078499],
       [-0.08460474,  0.0112464 ],
       [-0.08460474,  0.02168263],
       [-0.08460474,  0.0320071 ],
       [-0.08460474,  0.04226933],
       [-0.08460474,

In [47]:
packed_quant_embed = e["packed_embedding"]

In [48]:
n_samples = packed_quant_embed.shape[0]

In [49]:
lut[packed_quant_embed].reshape((n_samples, -1))

array([[-0.08460474,  0.12175898, -0.08460474, ...,  0.12175898,
        -0.08460474,  0.12175898],
       [-0.08460474,  0.12175898, -0.08460474, ...,  0.12175898,
        -0.08460474,  0.12175898],
       [-0.08460474,  0.12175898, -0.08460474, ...,  0.12175898,
        -0.08460474,  0.12175898],
       ...,
       [-0.08460474,  0.12175898, -0.08460474, ...,  0.12175898,
        -0.08460474,  0.12175898],
       [-0.08460474,  0.12175898, -0.08460474, ...,  0.12175898,
        -0.08460474,  0.12175898],
       [-0.08460474,  0.12175898, -0.08460474, ...,  0.12175898,
        -0.08460474,  0.12175898]], shape=(50000, 512))

In [50]:
e["embedding"].to_numpy()

array([[-0.15867826, -0.01054439, -0.00272848, ..., -0.01373062,
        -0.06487727,  0.04971681],
       [ 0.01134193,  0.03408474,  0.01067164, ..., -0.01303335,
        -0.00551121,  0.02126127],
       [ 0.03113855, -0.03197276,  0.06918616, ..., -0.06356741,
         0.02407873,  0.00431234],
       ...,
       [-0.10801465, -0.05486411, -0.01687499, ..., -0.1086157 ,
        -0.07798828, -0.02068592],
       [-0.08517856, -0.10697386, -0.04297443, ..., -0.04179037,
        -0.14975058, -0.01514342],
       [-0.05394588, -0.00101604,  0.01829   , ..., -0.09579103,
        -0.09284058, -0.09507327]], shape=(50000, 384), dtype=float32)

In [51]:
codebook["4bits"]

array([-0.12091945, -0.08460474, -0.06134046, -0.04448182, -0.03125859,
       -0.01997488, -0.00952856,  0.00078499,  0.0112464 ,  0.02168263,
        0.0320071 ,  0.04226933,  0.05339511,  0.06735399,  0.08780586,
        0.12175898])

In [52]:
e["packed_embedding"].to_numpy()

array([[31, 31, 31, ..., 31, 31, 31],
       [31, 31, 31, ..., 31, 31, 31],
       [31, 31, 31, ..., 31, 31, 31],
       ...,
       [31, 31, 31, ..., 31, 31, 31],
       [31, 31, 31, ..., 31, 31, 31],
       [31, 31, 31, ..., 31, 31, 31]], shape=(50000, 256), dtype=uint8)

In [55]:
e2 = pl.read_parquet("../data/processed/arxiv_quantized_embeddings_4bits.parquet")

In [56]:
e2

id,packed_embedding,embedding
str,"array[u8, 256]","array[f32, 384]"
"""0704.0001""","[31, 31, … 31]","[-0.158678, -0.010544, … 0.049717]"
"""0704.0002""","[31, 31, … 31]","[0.011342, 0.034085, … 0.021261]"
"""0704.0003""","[31, 31, … 31]","[0.031139, -0.031973, … 0.004312]"
"""0704.0004""","[31, 31, … 31]","[-0.068851, -0.018202, … 0.000149]"
"""0704.0005""","[31, 31, … 31]","[0.035553, 0.003334, … -0.026612]"
…,…,…
"""supr-con/9608008""","[31, 31, … 31]","[-0.105682, -0.014087, … -0.000596]"
"""supr-con/9609001""","[31, 31, … 31]","[-0.056112, 0.060603, … 0.068473]"
"""supr-con/9609002""","[31, 31, … 31]","[-0.027201, -0.00024, … 0.003988]"
